# 02 — Feature Assembly

This notebook doesn't add any new logic — everything it uses already lives in `src/preprocessing.py` and `src/features.py`, and is already tested in `tests/test_features.py`. Its only job is to:

1. Run the full pipeline chain: `load_and_prepare()` → `add_history_features()` → `drop_unlabeled()`, producing the final training-ready table.
2. Confirm its shape matches what we already proved it should be (4,315 rows).
3. Look at one real cable's full feature history, year by year, so the six computed history features are visible on an actual example instead of just living in test assertions.
4. Save the result to `outputs/features_labeled.csv`, so Week 3's modeling notebook can load one file instead of re-running this whole chain.

## Setup

This notebook lives in `notebooks/`, but `src/preprocessing.py` and `src/features.py` live one directory up, at the project root. `sys.path.append('..')` tells Python to also look in the project root when we write `from src... import ...`, the same way notebook 01 used `'../data/...'` to reach the data folder from inside `notebooks/`.

`config.DATA_PATH` (`'data/undersea_cables_master.csv'`) is written relative to the project root too, so we pass an explicit `'../data/undersea_cables_master.csv'` into `load_and_prepare()` rather than relying on its default.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd

from src.features import add_history_features
from src.preprocessing import drop_unlabeled, load_and_prepare

## Run the pipeline

Three steps, each already implemented and tested elsewhere:

- `load_and_prepare()` — load, sort, drop the 3 unusable columns, drop the 6 mislabeled gap rows. Keeps every row (including all 2026 rows) so history features can be computed for every cable.
- `add_history_features()` — adds `prev_fault`, `fault_3y`, `yrs_since_fault`, `never_faulted`, `cum_fault_rate`, `history_available`, year-aware across any gap.
- `drop_unlabeled()` — drops rows with no real `fault_next_year` to predict, producing the final training-ready table.

In [ ]:
prepared = load_and_prepare(path='../data/undersea_cables_master.csv')
featured = add_history_features(prepared)
labeled = drop_unlabeled(featured)

labeled.head()

## Confirm the shape

The row count (4,315) is a fact we've already established and tested — `drop_unlabeled()` asserts it internally, and `tests/test_features.py` checks it too, so we assert it again here as a third, visible confirmation. The column count isn't a fixed target the same way: it's whatever the original 21 columns minus the 3 dropped (`fault_cause`, `design_life_years`, `rfs_year`) plus the 6 new history features add up to — we print it rather than hard-code a number, since it would change if a feature were ever added or removed in `features.py`.

In [ ]:
print(f"Shape: {labeled.shape}")
print(f"Columns ({labeled.shape[1]}): {list(labeled.columns)}")

assert labeled.shape[0] == 4_315, f"Expected 4315 rows, got {labeled.shape[0]}"
print("PASS: row count matches the 4,315 established in Week 1 and tested in tests/test_features.py.")

## One cable's full feature history

`CAB0004` has a full 2015–2025 run (11 years, no gap) with faults in 2016, 2017, and 2021 — faults in the middle of its timeline, not just at the very start or end, which makes it a good example for seeing what each history feature actually does:

- `never_faulted` starts at `1` and flips to `0` the moment the first fault (2016) happens, then stays `0`.
- `yrs_since_fault` resets to `0` on a fault year, then counts up (`1, 2, 3, ...`) until the next fault.
- `fault_3y` picks up the back-to-back 2016–2017 faults as a 3-year sum of `2`, higher than any single-fault year.
- `cum_fault_rate` is the running fraction of all years so far that had a fault — it moves the most early on (few years to average over) and settles down later.
- `history_available` is `0` for the first three years (2015–2017, fewer than 3 *prior* years exist yet) and `1` from 2018 onward.

In [ ]:
# Without this, pandas truncates wide tables with a "..." in the middle --
# we want every column visible, not just the first and last few.
pd.set_option('display.max_columns', None)

cab0004 = labeled[labeled['cable_id'] == 'CAB0004'].sort_values('year')
cab0004

## Save for later notebooks

Per the blueprint's repo structure (§13), `outputs/` holds generated files so downstream notebooks and the app can just load a file instead of re-running the whole pipeline. `os.makedirs(..., exist_ok=True)` creates the `outputs/` folder if it doesn't exist yet, without raising an error if it already does. `index=False` keeps pandas from writing its own row-number column into the CSV — we don't want an extra, meaningless column showing up when this file gets read back in later.

In [ ]:
import os

os.makedirs('../outputs', exist_ok=True)
output_path = '../outputs/features_labeled.csv'
labeled.to_csv(output_path, index=False)

# Read it back to confirm the saved file actually matches what we just built,
# rather than just trusting that to_csv() worked.
reloaded = pd.read_csv(output_path)
print(f"Saved to {output_path}")
print(f"Reloaded shape: {reloaded.shape}")
assert reloaded.shape == labeled.shape, "Reloaded file doesn't match the dataframe we saved!"
print("PASS: saved file matches the in-memory dataframe.")